# In-Memory Tracer Stream

The `memory_stream.py` module implements an internal asynchronous memory channel used for communication between two coroutines.

The writer and reader may run in the same event loop or in different event loops on different threads. Data is stored in an unbounded `asyncio.Queue`, while writes are scheduled safely on the reader's event loop.

This module is internal to LangChain and does not expose a public API. Its internal classes are documented because they form the complete implementation.

## Type Variables

1. `T`: Represents the type of item transferred through the memory stream.
   * **Definition:**
     ```python
     T = TypeVar("T")
     ```

# _SendStream

`_SendStream` is the internal writing side of the memory channel.

It schedules queue writes on the event loop associated with the reader. This allows the writer to operate from the same thread or from a different thread and event loop.

## Bases

- `Generic[T]`

## Attributes

1. `_reader_loop`: Stores the event loop on which queue writes are scheduled.
   * **Type:**
     ```python
     _reader_loop: AbstractEventLoop
     ```

2. `_queue`: Stores the asynchronous queue shared with the receiving stream.
   * **Type:**
     ```python
     _queue: Queue[Any]
     ```

3. `_done`: Stores the sentinel object used to indicate that the stream has closed.
   * **Type:**
     ```python
     _done: object
     ```

### Methods

1. `__init__`: Creates a sending stream for a queue and closure sentinel.
   * **Syntax:**
     ```python
     __init__(
         self,
         reader_loop: AbstractEventLoop, # Event loop used to schedule queue writes
         queue: Queue[Any], # Queue receiving the transferred items
         done: object # Sentinel indicating that the writer has finished
     ) -> None
     ```

2. `send`: Asynchronously schedules an item to be written to the queue.

   The method delegates to `send_nowait`. It is asynchronous so callers may use an awaitable sending interface, but the queue operation itself is scheduled without waiting for completion.

   * **Syntax:**
     ```python
     async send(
         self,
         item: T # Item to write to the stream
     ) -> None
     ```

3. `send_nowait`: Schedules an item on the reader's event loop without blocking.

   The method uses `AbstractEventLoop.call_soon_threadsafe` to invoke `Queue.put_nowait`, allowing writes from another thread.

   A `RuntimeError` is re-raised when scheduling fails while the reader loop is still open. The error is suppressed when the loop has already closed.

   * **Syntax:**
     ```python
     send_nowait(
         self,
         item: T # Item to write to the stream
     ) -> None
     ```

4. `aclose`: Asynchronously closes the sending stream.

   The method delegates to `close` and schedules the sentinel object to be written to the queue.

   * **Syntax:**
     ```python
     async aclose(
         self
     ) -> None
     ```

5. `close`: Closes the sending stream without blocking.

   The sentinel object is scheduled on the reader's event loop through `call_soon_threadsafe`. When the receiver reads this sentinel, it stops iteration.

   A `RuntimeError` is re-raised when scheduling fails while the reader loop is still open. The error is suppressed when the loop has already closed.

   * **Syntax:**
     ```python
     close(
         self
     ) -> None
     ```

# _ReceiveStream

`_ReceiveStream` is the internal reading side of the memory channel.

It is an asynchronous iterable that waits for queue items and yields them until it receives the shared closure sentinel.

The receiver is expected to run in the same event loop that was supplied when creating `_MemoryStream`.

## Bases

- `Generic[T]`

## Attributes

1. `_queue`: Stores the asynchronous queue shared with the sending stream.
   * **Type:**
     ```python
     _queue: Queue[Any]
     ```

2. `_done`: Stores the sentinel object that terminates iteration.
   * **Type:**
     ```python
     _done: object
     ```

3. `_is_closed`: Indicates whether the closure sentinel has been received.
   * **Type:**
     ```python
     _is_closed: bool
     ```

### Methods

1. `__init__`: Creates a receiving stream for a queue and closure sentinel.
   * **Syntax:**
     ```python
     __init__(
         self,
         queue: Queue[Any], # Queue from which items are received
         done: object # Sentinel indicating that the writer has finished
     ) -> None
     ```

2. `__aiter__`: Asynchronously yields items from the queue until the closure sentinel is received.

   When the sentinel is encountered, `_is_closed` is set to `True`, the sentinel is not yielded, and iteration ends.

   * **Syntax:**
     ```python
     async __aiter__(
         self
     ) -> AsyncIterator[T]
     ```

# _MemoryStream

`_MemoryStream` is the internal channel that creates connected sending and receiving streams.

It uses one unbounded `asyncio.Queue` and one unique sentinel object. The implementation is intended for a single writer and a single reader.

The reader is assumed to run on the event loop supplied to the constructor. This requirement is not validated at runtime.

## Bases

- `Generic[T]`

## Attributes

1. `_loop`: Stores the event loop associated with the receiving side.
   * **Type:**
     ```python
     _loop: AbstractEventLoop
     ```

2. `_queue`: Stores the unbounded queue used to transfer items.
   * **Type:**
     ```python
     _queue: Queue[Any]
     ```

3. `_done`: Stores the unique sentinel used to signal stream completion.
   * **Type:**
     ```python
     _done: object
     ```

### Methods

1. `__init__`: Creates a memory channel for a reader running on the supplied event loop.

   The queue has `maxsize=0`, making it unbounded.

   * **Syntax:**
     ```python
     __init__(
         self,
         loop: AbstractEventLoop # Event loop on which the reader operates
     ) -> None
     ```

2. `get_send_stream`: Returns a sending stream connected to this channel.

   The returned writer shares the channel's event loop, queue, and closure sentinel.

   * **Syntax:**
     ```python
     get_send_stream(
         self
     ) -> _SendStream[T]
     ```

3. `get_receive_stream`: Returns a receiving stream connected to this channel.

   The returned reader shares the channel's queue and closure sentinel.

   * **Syntax:**
     ```python
     get_receive_stream(
         self
     ) -> _ReceiveStream[T]
     ```

## Communication Behaviour

The channel follows this lifecycle:

1. `_MemoryStream` creates one unbounded queue and one sentinel.
2. `_SendStream` schedules items through the reader's event loop.
3. `_ReceiveStream` awaits and yields those queue items.
4. Closing `_SendStream` places the sentinel into the queue.
5. `_ReceiveStream` detects the sentinel, marks itself closed, and ends iteration.

Because scheduling uses `call_soon_threadsafe`, the writer may operate from a different thread or event loop than the reader.